# Ejemplo o caso práctico de aplicación de arquitectura kappa con PySpark

Bienvenido a este cuaderno interactivo diseñado desde la perspectiva de la ingeniería de datos en entornos de Big Data. En esta guía construiremos paso a paso un flujo de datos implementando el patrón de **Arquitectura Kappa** utilizando **Apache Spark (PySpark) y Structured Streaming**.

---

## 🛠️ Guía paso a paso: Cómo importar y ejecutar este cuaderno (`.ipynb`)

Este laboratorio está diseñado y verificado para ejecutarse de principio a fin tanto en **Databricks Free / Community Edition** como en **Google Colab** o entornos locales, utilizando un volumen de Unity Catalog y procesamiento de streams con control de estado (*Checkpointing*) sin requerir servidores externos de Kafka.

### Opción A: Ejecutar en Databricks Free / Community Edition
1. Inicia sesión en tu cuenta de **Databricks**.
2. En el menú lateral izquierdo, haz clic en **Workspace** (Área de trabajo).
3. Navega a tu carpeta de usuario, haz **clic derecho** en un espacio vacío y selecciona **Import** (Importar).
4. Arrastra o selecciona este archivo `arquitectura_kappa_pyspark.ipynb` desde tu ordenador y haz clic en **Import**.
5. En la esquina superior derecha del cuaderno, conecta tu **Cluster** (Compute) activo y ejecuta las celdas en orden usando `Shift + Enter`.

### Opción B: Alternativa gratuita recomendada (Google Colab)
Si prefieres no esperar al arranque del clúster de Databricks Community Edition:
1. Entra en [Google Colab](https://colab.research.google.com/) con tu cuenta de Google.
2. En el menú **Archivo > Subir cuaderno**, selecciona este archivo `.ipynb`.
3. Ejecuta las celdas normalmente: **la primera celda de código detectará automáticamente que estás fuera de Databricks e instalará e inicializará PySpark por su cuenta**.

---

## 1. ¿Qué es la Arquitectura Kappa y en qué se diferencia de Lambda?

En el caso práctico anterior vimos que la **Arquitectura Lambda** mantenía dos caminos separados: una **Capa Batch** (para precisión histórica) y una **Capa Speed** (para baja latencia). Aunque efectiva, Lambda tiene un gran inconveniente en ingeniería de datos: **obliga a programar, probar y mantener la misma lógica de negocio en dos lugares distintos**.

En 2014, **Jay Kreps** (cocreador de Apache Kafka) propuso la **Arquitectura Kappa** bajo una premisa revolucionaria:
> *"¿Por qué no tratamos TODOS los datos como un flujo continuo (Stream) y eliminamos por completo la Capa Batch?"*

### Los 3 pilares de la Arquitectura Kappa:
1. **Registro de Eventos Inmutable (Append-Only Log):**
   * Todos los eventos (históricos y nuevos) se almacenan en orden cronológico en un sistema de colas o registro de eventos (como Apache Kafka, Event Hubs o un directorio de eventos inmutables en un Data Lake).
2. **Una Única Capa de Procesamiento (Stream Processing):**
   * Solo existe **un único código** escrito con un motor de streaming (en nuestro caso, `Spark Structured Streaming`). Este motor lee los eventos a medida que llegan y actualiza las vistas de negocio en tiempo real.
3. **Reprocesamiento Histórico (Replay / Recomputación):**
   * ¿Qué ocurre si cambiamos la lógica de negocio o descubrimos un error en nuestra fórmula? En lugar de usar un proceso Batch separado, simplemente **borramos el estado de control (*Checkpoint*) y le pedimos a nuestro motor de Streaming que vuelva a leer el registro de eventos desde el principio (*Offset 0*)** aplicando la nueva lógica a máxima velocidad.

> ### 🔧 Versión adaptada a Databricks Free Edition (Serverless)
> Esta versión del cuaderno corrige lo que fallaba en Serverless:
> 1. **Rutas**: en vez de `/tmp/...` (DBFS raíz, bloqueado en Free Edition) se usa un **volumen de Unity Catalog** (`/Volumes/workspace/default/...`), accesible igual desde Spark y desde Python (`open`, `os`).
> 2. **Streaming**: Serverless solo admite `trigger(availableNow=True)` (procesa todo lo pendiente y termina) y no admite el sink `memory`. Por eso cada ejecución del stream usa `foreachBatch` para guardar la vista en Parquet, y se lanza con `ejecutar_...()` cada vez que llegan datos. El `checkpointLocation` hace que cada ejecución solo procese los ficheros nuevos, así que **la lógica de la arquitectura no cambia**.
> 3. Sigue funcionando sin cambios en Google Colab / local (detecta el entorno automáticamente).

---
## Paso 0: Preparación del Entorno y Sesión de Spark

En esta primera celda verificamos el entorno (Databricks o Google Colab/Local), inicializamos la sesión `spark` y preparamos las carpetas que actuarán como nuestro **Registro Inmutable de Eventos (Event Log)** y nuestros directorios de **Checkpointing** (donde Spark guarda qué eventos ya ha leído).

In [ ]:
import os
import json
import time
import shutil
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Detectamos el entorno: Databricks (Serverless / Free Edition) o Colab / local
try:
    spark
    EN_DATABRICKS = "dbutils" in globals()
    print("✅ Sesión 'spark' ya activa" + (" (Databricks)." if EN_DATABRICKS else "."))
except NameError:
    EN_DATABRICKS = False
    print("⚙️ No se detectó sesión previa de Spark. Configurando PySpark (Modo Google Colab / Local)...")
    try:
        import pyspark
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "pyspark"], check=True)
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName("KappaArchitecture_PySpark") \
        .master("local[*]") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()
    print("✅ Sesión de Spark creada con éxito.")

# 2. Rutas de trabajo.
#    Databricks Free Edition NO permite escribir en la raíz de DBFS (/tmp/... falla o no coincide
#    entre Python y Spark), así que usamos un VOLUMEN de Unity Catalog: /Volumes/<catalogo>/<esquema>/<volumen>
if EN_DATABRICKS:
    CATALOGO, ESQUEMA, VOLUMEN = "workspace", "default", "lambda_kappa_lab"
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")
    BASE_PATH = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}/kappa_ecommerce_lab"
else:
    BASE_PATH = "/tmp/kappa_ecommerce_lab"

def borrar_ruta(ruta):
    """Borra una carpeta de forma compatible con Databricks (volúmenes) y con local/Colab."""
    if EN_DATABRICKS:
        dbutils.fs.rm(ruta, True)
    elif os.path.exists(ruta):
        shutil.rmtree(ruta)

# Detenemos cualquier stream previo que haya quedado activo en la sesión por seguridad
try:
    for s in spark.streams.active:
        s.stop()
except Exception:
    pass

EVENT_LOG_PATH = f"{BASE_PATH}/immutable_event_log"   # Nuestro 'Kafka' o registro inmutable
CHECKPOINT_V1_PATH = f"{BASE_PATH}/checkpoint_v1"     # Estado del procesador versión 1
CHECKPOINT_V2_PATH = f"{BASE_PATH}/checkpoint_v2"     # Estado para el reprocesamiento (Replay)
VISTA_V1_PATH = f"{BASE_PATH}/serving_kappa_v1"       # Vista de servicio V1
VISTA_V2_PATH = f"{BASE_PATH}/serving_kappa_v2"       # Vista de servicio V2

# Limpiamos directorios previos para garantizar una ejecución limpia desde cero
borrar_ruta(BASE_PATH)

os.makedirs(EVENT_LOG_PATH, exist_ok=True)

print(f"📁 Entorno Kappa inicializado en: {BASE_PATH}")

---
## Caso de Uso: Analítica Continua de E-Commerce ("TechStore")

Para que puedas comparar directamente la **Arquitectura Kappa** con la **Arquitectura Lambda**, utilizaremos el mismo escenario de la tienda **TechStore** (`Laptops`, `Smartphones`, `Audio`, `Monitores`, `Gaming`).

En lugar de tener una tabla histórica en Parquet por un lado y JSONs por otro:
* **Todo** (tanto las compras de ayer como las de hoy) vive en una única secuencia cronológica dentro de `EVENT_LOG_PATH`.
* Crearemos una función auxiliar para inyectar lotes de eventos en este registro inmutable.

In [ ]:
# 1. Esquema estricto de los eventos de compra
esquema_eventos = StructType([
    StructField("id_transaccion", StringType(), False),
    StructField("categoria", StringType(), False),
    StructField("monto", DoubleType(), False),
    StructField("fecha_hora", StringType(), False)
])

# Función auxiliar para guardar eventos en nuestro registro inmutable (Append-Only Log)
def publicar_eventos_en_log(nombre_archivo, lista_eventos):
    ruta_archivo = os.path.join(EVENT_LOG_PATH, nombre_archivo)
    with open(ruta_archivo, "w", encoding="utf-8") as f:
        for ev in lista_eventos:
            f.write(json.dumps(ev) + "\n")
    print(f"📥 Publicados {len(lista_eventos)} eventos en el log: {nombre_archivo}")

# 2. Publicamos en el log los eventos iniciales (Offset 0: Histórico del día anterior)
eventos_bloque_0 = [
    {"id_transaccion": "TX_001", "categoria": "Laptops", "monto": 1200.00, "fecha_hora": "2026-05-01 10:15:00"},
    {"id_transaccion": "TX_002", "categoria": "Smartphones", "monto": 800.00, "fecha_hora": "2026-05-01 11:20:00"},
    {"id_transaccion": "TX_003", "categoria": "Audio", "monto": 150.00, "fecha_hora": "2026-05-01 12:05:00"},
    {"id_transaccion": "TX_004", "categoria": "Laptops", "monto": 1500.00, "fecha_hora": "2026-05-01 14:30:00"},
    {"id_transaccion": "TX_005", "categoria": "Monitores", "monto": 400.00, "fecha_hora": "2026-05-01 15:45:00"},
    {"id_transaccion": "TX_006", "categoria": "Smartphones", "monto": 950.00, "fecha_hora": "2026-05-01 16:10:00"},
    {"id_transaccion": "TX_007", "categoria": "Audio", "monto": 200.00, "fecha_hora": "2026-05-01 17:50:00"},
    {"id_transaccion": "TX_008", "categoria": "Monitores", "monto": 350.00, "fecha_hora": "2026-05-01 18:20:00"},
    {"id_transaccion": "TX_009", "categoria": "Laptops", "monto": 1100.00, "fecha_hora": "2026-05-01 19:00:00"},
    {"id_transaccion": "TX_010", "categoria": "Smartphones", "monto": 750.00, "fecha_hora": "2026-05-01 20:15:00"}
]

publicar_eventos_en_log("log_000_historico.json", eventos_bloque_0)

---
## Paso 2: Capa Única de Streaming (El corazón de la Arquitectura Kappa)

### Teoría rápida:
En Kappa **no escribimos un Job Batch y un Job Streaming**. Escribimos **una sola tubería de Streaming (`pipeline_kappa_v1`)** que lee desde `EVENT_LOG_PATH`.
* Al arrancar por primera vez, el stream procesa todos los eventos históricos existentes en el log (`log_000_historico.json`).
* Inmediatamente después, se queda escuchando nuevos eventos en tiempo real sin necesidad de hacer ningún `FULL OUTER JOIN` entre capas.

In [ ]:
# 1. Leemos el flujo continuo desde nuestro Registro Inmutable de Eventos
df_stream_log = spark.readStream \
    .schema(esquema_eventos) \
    .option("maxFilesPerTrigger", 1) \
    .json(EVENT_LOG_PATH) \
    .withColumn("fecha_hora", F.to_timestamp("fecha_hora"))

# 2. Lógica de Negocio V1 (Única capa de código): Agregación directa sobre el stream
df_vista_kappa_v1 = df_stream_log.groupBy("categoria").agg(
    F.count("id_transaccion").alias("total_transacciones"),
    F.round(F.sum("monto"), 2).alias("ingresos_brutos"),
    F.max("fecha_hora").alias("ultimo_evento_registrado")
)

# 3. Función que ejecuta un pipeline de streaming con Trigger.AvailableNow:
#    procesa todos los eventos pendientes del log (según su checkpoint) y termina.
#    foreachBatch guarda la vista de servicio completa en Parquet (Serverless no admite el sink 'memory').
def ejecutar_pipeline(df_vista, ruta_vista, ruta_checkpoint):
    def guardar_vista(batch_df, batch_id):
        batch_df.write.mode("overwrite").parquet(ruta_vista)
    q = (df_vista.writeStream
         .outputMode("complete")
         .foreachBatch(guardar_vista)
         .option("checkpointLocation", ruta_checkpoint)
         .trigger(availableNow=True)
         .start())
    q.awaitTermination()

# Procesamos el bloque inicial disponible en el log
ejecutar_pipeline(df_vista_kappa_v1, VISTA_V1_PATH, CHECKPOINT_V1_PATH)

print("📊 Vista Serving Kappa V1 (tras procesar el inicio del log):")
spark.read.parquet(VISTA_V1_PATH).orderBy(F.desc("ingresos_brutos")).show(truncate=False)

---
## Paso 3: Llegada de Eventos en Tiempo Real

Veamos qué ocurre cuando los clientes realizan nuevas compras hoy. Simplemente se añaden nuevos eventos al registro inmutable (`EVENT_LOG_PATH`) y **la misma tubería de streaming** actualiza la tabla `serving_kappa_v1` automáticamente.

In [ ]:
# Llegan dos nuevos bloques de compras en vivo durante la mañana de hoy
eventos_bloque_1 = [
    {"id_transaccion": "TX_101", "categoria": "Laptops", "monto": 1350.00, "fecha_hora": "2026-05-02 09:05:00"},
    {"id_transaccion": "TX_102", "categoria": "Audio", "monto": 300.00, "fecha_hora": "2026-05-02 09:06:15"},
    {"id_transaccion": "TX_103", "categoria": "Audio", "monto": 120.00, "fecha_hora": "2026-05-02 09:08:40"}
]
publicar_eventos_en_log("log_001_vivo_manana.json", eventos_bloque_1)

eventos_bloque_2 = [
    {"id_transaccion": "TX_104", "categoria": "Smartphones", "monto": 900.00, "fecha_hora": "2026-05-02 09:15:00"},
    {"id_transaccion": "TX_105", "categoria": "Gaming", "monto": 600.00, "fecha_hora": "2026-05-02 09:18:22"}
]
publicar_eventos_en_log("log_002_vivo_mediodia.json", eventos_bloque_2)

# Relanzamos la MISMA tubería: gracias al Checkpoint solo procesa los archivos nuevos
ejecutar_pipeline(df_vista_kappa_v1, VISTA_V1_PATH, CHECKPOINT_V1_PATH)

print("⚡ Vista Serving Kappa V1 actualizada (Histórico + Hoy unificados sin Joins):")
spark.read.parquet(VISTA_V1_PATH).orderBy(F.desc("ingresos_brutos")).show(truncate=False)

---
## Paso 4: El Superpoder de Kappa — Reprocesamiento Histórico (*Stream Replay*)

### ¿Qué pasa si cambia la regla de negocio o hay que corregir un cálculo?
Imagina que el departamento financiero nos avisa:
> *"¡Atención Ingeniería de Datos! Todos los montos registrados en los eventos incluyen un 21% de impuestos (IVA) y necesitamos que el dashboard muestre también los **Ingresos Netos (sin impuestos)** y el **Ticket Promedio** desde el primer día."*

* En una **Arquitectura Lambda**, tendrías que modificar dos programas (el Job Batch y el Job Streaming) y esperar a la sincronización.
* En la **Arquitectura Kappa**, como todos los eventos crudos siguen intactos en nuestro registro inmutable (`EVENT_LOG_PATH`), simplemente:
  1. Lanzamos la **Versión 2** de nuestro procesador de Streaming apuntando a un nuevo Checkpoint limpio (`CHECKPOINT_V2_PATH`).
  2. Spark lee automáticamente **toda la historia desde el primer evento (`log_000`) hasta el último (`log_002`)** aplicando la nueva fórmula en segundos.
  3. Cuando la vista V2 está lista, apagamos la V1.

In [ ]:
# 1. Definimos la nueva lectura desde el principio del Event Log
df_stream_replay = spark.readStream \
    .schema(esquema_eventos) \
    .json(EVENT_LOG_PATH) \
    .withColumn("fecha_hora", F.to_timestamp("fecha_hora"))

# 2. Nueva Lógica de Negocio V2: Añadimos Ingresos Netos (descontando 21% IVA) y Ticket Promedio
df_vista_kappa_v2 = df_stream_replay \
    .withColumn("monto_neto", F.col("monto") / F.lit(1.21)) \
    .groupBy("categoria").agg(
        F.count("id_transaccion").alias("total_transacciones"),
        F.round(F.sum("monto"), 2).alias("ingresos_brutos"),
        F.round(F.sum("monto_neto"), 2).alias("ingresos_netos_sin_iva"),
        F.round(F.avg("monto"), 2).alias("ticket_promedio"),
        F.max("fecha_hora").alias("ultimo_evento_registrado")
    )

# 3. Lanzamos el pipeline V2 con un Checkpoint nuevo (esto fuerza a leer desde el Offset 0)
ejecutar_pipeline(df_vista_kappa_v2, VISTA_V2_PATH, CHECKPOINT_V2_PATH)

print("🔄 REPROCESAMIENTO (REPLAY) COMPLETADO — Nueva Vista Serving Kappa V2:")
spark.read.parquet(VISTA_V2_PATH).orderBy(F.desc("ingresos_brutos")).show(truncate=False)

# 4. Con Trigger.AvailableNow cada ejecución termina sola: no hay streams que detener.
#    En un entorno real, cuando V2 esté validada se apagaría el pipeline V1 y se redirigirían las consultas a V2.
print("✅ Pipelines V1 y V2 finalizados. Sin streams activos ni recursos que liberar.")

---
## 5. Comparativa Final: ¿Cuándo usar Lambda vs. Kappa?
Ahora conoces de forma práctica las dos arquitecturas clásicas más importantes del procesamiento de datos en Big Data.

| Característica | Arquitectura Lambda | Arquitectura Kappa |
| :--- | :--- | :--- |
| **Capas de código** | **2 capas** (Batch + Speed). Hay que mantener doble código. | **1 única capa** (Todo es Streaming). |
| **Complejidad en Capa Serving** | Alta (requiere `FULL OUTER JOIN` entre vistas Batch y Speed). | Baja (la vista ya está unificada por el motor de Streaming). |
| **Cómo recalcula la historia** | Re-ejecuta el Job Batch sobre el *Master Dataset*. | Hace **Replay** leyendo el registro de eventos desde el inicio (*Offset 0*). |
| **Caso ideal** | Cuando los algoritmos Batch (ej. Machine Learning complejo o grafos pesados) son imposibles de expresar en Streaming. | Cuando las transformaciones y agregaciones son idénticas para datos históricos y datos en tiempo real. |